In [ ]:
from pathlib import Path
from collections import defaultdict
import sys
import warnings

import numpy as np
import pandas as pd
import tracksdata as td

# =========================================================
# Project paths
# =========================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_DIR = (
    PROJECT
    / "data/raw/competition/"
    / "biohub-cell-tracking-during-development/train"
)

PRED_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0_det09950"
    / "split_0"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)

OFFICIAL_SRC = (
    PROJECT
    / "external"
    / "official"
    / "src"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external"
    / "official"
    / "scripts"
)

for p in [OFFICIAL_SRC, OFFICIAL_SCRIPTS]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

warnings.filterwarnings(
    "ignore",
    message="Graph already matched, overwriting previous matching."
)

# =========================================================
# Frozen baseline-v3 configuration
# =========================================================

DET_THRESHOLD = 0.995

# Frozen G2 division policy
G2_P2_MIN = 0.55
G2_DAUGHTER_SEP_MIN_UM = 6.0
G2_ANGLE_MIN_DEG = 120.0
G2_MAX_PARENT_DAUGHTER_UM = 8.0
G2_DISTANCE_BALANCE_MAX = 0.50

# Frozen component cleanup
TWO_NODE_MAX_DIST_UM = 3.5

FROZEN_FOLD0_SCORE = 0.741854

# =========================================================
# CV metadata
# =========================================================

cv_meta = pd.read_csv(CV_PATH)

fold0 = (
    cv_meta[
        cv_meta["fold"] == 0
    ]
    .copy()
)

prefix_map = dict(
    zip(
        fold0["dataset"],
        fold0["prefix"],
    )
)

estimated_map = dict(
    zip(
        fold0["dataset"],
        fold0["estimated_nodes"],
    )
)

# =========================================================
# Helpers
# =========================================================

def load_graph(path):
    loaded = td.graph.IndexedRXGraph.from_geff(path)
    return (
        loaded[0]
        if isinstance(loaded, tuple)
        else loaded
    )


def apply_frozen_g2(graph, scale):
    """
    Frozen G2 division filter.

    For every predicted fork:
      - keep highest-probability child
      - keep second child only if G2 geometry passes
    """

    scale = np.asarray(
        scale,
        dtype=float,
    )

    nodes = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        graph
        .edge_attrs(
            attr_keys=["edge_prob"],
            unpack=True,
        )
        .to_pandas()
    )

    remove_ids = []
    forks_kept = 0

    for source_id, group in edges.groupby(
        "source_id"
    ):

        if len(group) != 2:
            continue

        group = group.sort_values(
            ["edge_prob", "edge_id"],
            ascending=[False, True],
        )

        e1 = group.iloc[0]
        e2 = group.iloc[1]

        source_id = int(source_id)
        target1 = int(e1["target_id"])
        target2 = int(e2["target_id"])

        parent = nodes.loc[source_id]
        child1 = nodes.loc[target1]
        child2 = nodes.loc[target2]

        p = (
            np.array(
                [parent["z"], parent["y"], parent["x"]],
                dtype=float,
            )
            * scale
        )

        c1 = (
            np.array(
                [child1["z"], child1["y"], child1["x"]],
                dtype=float,
            )
            * scale
        )

        c2 = (
            np.array(
                [child2["z"], child2["y"], child2["x"]],
                dtype=float,
            )
            * scale
        )

        v1 = c1 - p
        v2 = c2 - p

        d1 = float(np.linalg.norm(v1))
        d2 = float(np.linalg.norm(v2))

        daughter_sep = float(
            np.linalg.norm(c1 - c2)
        )

        if d1 > 0 and d2 > 0:
            cosine = float(
                np.dot(v1, v2)
                / (d1 * d2)
            )

            cosine = float(
                np.clip(
                    cosine,
                    -1.0,
                    1.0,
                )
            )

            angle = float(
                np.degrees(
                    np.arccos(cosine)
                )
            )
        else:
            angle = np.nan

        distance_balance = (
            abs(d1 - d2)
            / max(d1 + d2, 1e-12)
        )

        keep_second = (
            float(e2["edge_prob"]) >= G2_P2_MIN

            and daughter_sep
            >= G2_DAUGHTER_SEP_MIN_UM

            and np.isfinite(angle)

            and angle >= G2_ANGLE_MIN_DEG

            and max(d1, d2)
            <= G2_MAX_PARENT_DAUGHTER_UM

            and distance_balance
            <= G2_DISTANCE_BALANCE_MAX
        )

        if keep_second:
            forks_kept += 1
        else:
            remove_ids.append(
                int(e2["edge_id"])
            )

    if remove_ids:
        graph.bulk_remove_edges(
            remove_ids
        )

    return forks_kept


# =========================================================
# Preflight
# =========================================================

pred_files = sorted(
    PRED_DIR.glob("*.geff")
)

print("Prediction directory:", PRED_DIR)
print("Prediction GEFFs:", len(pred_files))
print("Fold 0 datasets:", len(fold0))
print("Detection threshold:", DET_THRESHOLD)
print("Frozen Fold 0 score:", FROZEN_FOLD0_SCORE)

assert len(pred_files) == 40
assert len(fold0) == 40

assert set(
    p.stem for p in pred_files
) == set(
    fold0["dataset"]
)

print("\n07 association audit setup: OK")

In [ ]:
association_rows = []

pred_files = sorted(
    PRED_DIR.glob("*.geff")
)

for i, pred_path in enumerate(
    pred_files,
    start=1,
):
    name = pred_path.stem

    pred_g = load_graph(
        pred_path
    )

    gt_g = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = np.asarray(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    # =====================================================
    # 1. Apply frozen G2
    # =====================================================

    apply_frozen_g2(
        pred_g,
        scale,
    )

    # =====================================================
    # 2. Official node matching
    # =====================================================

    _ = official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(scale),
        max_distance=7.0,
    )

    nodes = (
        pred_g
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        pred_g
        .edge_attrs(
            attr_keys=[
                "edge_prob",
                "edge_dist",
            ],
            unpack=True,
        )
        .to_pandas()
    )

    gt_edge_set = {
        (int(source), int(target))
        for source, target
        in gt_g.edge_list()
    }

    # =====================================================
    # 3. Degree information after G2
    # =====================================================

    outdegree = (
        edges
        .groupby("source_id")
        .size()
        .to_dict()
    )

    indegree = (
        edges
        .groupby("target_id")
        .size()
        .to_dict()
    )

    # =====================================================
    # 4. Build edge-level diagnostic table
    # =====================================================

    for _, e in edges.iterrows():

        source = int(
            e["source_id"]
        )

        target = int(
            e["target_id"]
        )

        a = nodes.loc[source]
        b = nodes.loc[target]

        source_match = int(
            a["match_node_id"]
        )

        target_match = int(
            b["match_node_id"]
        )

        both_nodes_matched = (
            source_match >= 0
            and target_match >= 0
        )

        direct_true_edge = (
            both_nodes_matched
            and (
                source_match,
                target_match,
            ) in gt_edge_set
        )

        delta_um = (
            np.array(
                [
                    float(b["z"] - a["z"]),
                    float(b["y"] - a["y"]),
                    float(b["x"] - a["x"]),
                ]
            )
            * scale
        )

        physical_dist_um = float(
            np.linalg.norm(
                delta_um
            )
        )

        association_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "source_id":
                source,

            "target_id":
                target,

            "edge_prob":
                float(
                    e["edge_prob"]
                ),

            "edge_dist_raw":
                float(
                    e["edge_dist"]
                ),

            "physical_dist_um":
                physical_dist_um,

            "delta_t":
                int(
                    b["t"] - a["t"]
                ),

            "source_outdegree":
                int(
                    outdegree.get(
                        source,
                        0,
                    )
                ),

            "target_indegree":
                int(
                    indegree.get(
                        target,
                        0,
                    )
                ),

            "source_matched":
                source_match >= 0,

            "target_matched":
                target_match >= 0,

            "both_nodes_matched":
                both_nodes_matched,

            "direct_true_edge":
                direct_true_edge,
        })

    if i % 10 == 0:
        print(
            f"Processed {i}/40"
        )


association_edge_df = pd.DataFrame(
    association_rows
)

print(
    "\nEdges after G2:",
    len(association_edge_df),
)

print(
    "Direct true edges:",
    int(
        association_edge_df[
            "direct_true_edge"
        ].sum()
    ),
)

print(
    "Both nodes matched:",
    int(
        association_edge_df[
            "both_nodes_matched"
        ].sum()
    ),
)


# =========================================================
# Ordinary non-division edges only
# =========================================================

ordinary_edges = (
    association_edge_df[
        association_edge_df[
            "source_outdegree"
        ] == 1
    ]
    .copy()
)

print(
    "\nOrdinary association edges:",
    len(ordinary_edges),
)

print(
    "Direct true ordinary edges:",
    int(
        ordinary_edges[
            "direct_true_edge"
        ].sum()
    ),
)


# =========================================================
# Diagnostic distributions
# =========================================================

quantiles = [
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
]


print(
    "\n=== EDGE PROB: TRUE vs OTHER ==="
)

display(
    ordinary_edges
    .groupby(
        "direct_true_edge"
    )[
        "edge_prob"
    ]
    .quantile(
        quantiles
    )
    .unstack()
)


print(
    "\n=== PHYSICAL DISTANCE: TRUE vs OTHER ==="
)

display(
    ordinary_edges
    .groupby(
        "direct_true_edge"
    )[
        "physical_dist_um"
    ]
    .quantile(
        quantiles
    )
    .unstack()
)


print(
    "\n=== BASIC SUMMARY ==="
)

display(
    ordinary_edges
    .groupby(
        "direct_true_edge"
    )[
        [
            "edge_prob",
            "physical_dist_um",
            "both_nodes_matched",
        ]
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
        ]
    )
)


print(
    "\n=== DELTA-T CHECK ==="
)

display(
    ordinary_edges[
        "delta_t"
    ]
    .value_counts()
    .sort_index()
)

1. Detection / node quality
2. Track/component quality
3. Division
4. Association choice

In [ ]:
# =========================================================
# Component-level quality audit
#
# Frozen:
#   det = 0.995
#   G2 division filter
#
# Diagnostic GT labels only.
# =========================================================
# =========================================================
# Local helper: weakly connected components
# Keep inside this cell so it does not depend on notebook state
# =========================================================

from collections import defaultdict


def component_sets(graph):
    node_ids = [
        int(x)
        for x in graph.node_attrs(
            unpack=True
        )["node_id"].to_list()
    ]

    parent = {
        n: n
        for n in node_ids
    }

    size = {
        n: 1
        for n in node_ids
    }

    def find(x):
        root = x

        while parent[root] != root:
            root = parent[root]

        while parent[x] != x:
            nxt = parent[x]
            parent[x] = root
            x = nxt

        return root

    def union(a, b):
        ra = find(a)
        rb = find(b)

        if ra == rb:
            return

        if size[ra] < size[rb]:
            ra, rb = rb, ra

        parent[rb] = ra
        size[ra] += size[rb]

    for source, target in graph.edge_list():
        union(
            int(source),
            int(target),
        )

    components = defaultdict(list)

    for node_id in node_ids:
        components[
            find(node_id)
        ].append(node_id)

    return list(
        components.values()
    )


print(
    "component_sets defined:",
    "component_sets" in globals()
)
component_rows = []

pred_files = sorted(
    PRED_DIR.glob("*.geff")
)

for i, pred_path in enumerate(
    pred_files,
    start=1,
):
    name = pred_path.stem

    pred_g = load_graph(
        pred_path
    )

    gt_g = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = np.asarray(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    # ---------------------------------------------
    # Frozen G2
    # ---------------------------------------------

    apply_frozen_g2(
        pred_g,
        scale,
    )

    # ---------------------------------------------
    # Official matching for diagnostic labels
    # ---------------------------------------------

    _ = official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(scale),
        max_distance=7.0,
    )

    nodes = (
        pred_g
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        pred_g
        .edge_attrs(
            attr_keys=[
                "edge_prob",
                "edge_dist",
            ],
            unpack=True,
        )
        .to_pandas()
    )

    gt_edge_set = {
        (int(s), int(t))
        for s, t in gt_g.edge_list()
    }

    # ---------------------------------------------
    # Edge lookup
    # ---------------------------------------------

    edges_by_node = defaultdict(list)

    for _, e in edges.iterrows():

        s = int(e["source_id"])
        t = int(e["target_id"])

        edges_by_node[s].append(e)
        edges_by_node[t].append(e)

    # ---------------------------------------------
    # Components
    # ---------------------------------------------

    components = component_sets(
        pred_g
    )

    for comp in components:

        comp = list(
            map(int, comp)
        )

        comp_set = set(comp)

        comp_nodes = nodes.loc[
            comp
        ]

        # Unique internal edges
        comp_edges = edges[
            edges["source_id"].isin(comp_set)
            & edges["target_id"].isin(comp_set)
        ]

        n_nodes = len(comp)
        n_edges = len(comp_edges)

        t_min = int(
            comp_nodes["t"].min()
        )

        t_max = int(
            comp_nodes["t"].max()
        )

        temporal_span = (
            t_max - t_min + 1
        )

        # -----------------------------------------
        # Edge probability statistics
        # -----------------------------------------

        if n_edges > 0:

            edge_probs = (
                comp_edges[
                    "edge_prob"
                ]
                .to_numpy(float)
            )

            mean_edge_prob = float(
                edge_probs.mean()
            )

            median_edge_prob = float(
                np.median(edge_probs)
            )

            min_edge_prob = float(
                edge_probs.min()
            )

        else:

            mean_edge_prob = np.nan
            median_edge_prob = np.nan
            min_edge_prob = np.nan

        # -----------------------------------------
        # Physical movement
        # -----------------------------------------

        physical_distances = []

        direct_true_edges = 0

        for _, e in comp_edges.iterrows():

            source = int(
                e["source_id"]
            )

            target = int(
                e["target_id"]
            )

            a = nodes.loc[source]
            b = nodes.loc[target]

            delta_um = (
                np.array([
                    float(
                        b["z"] - a["z"]
                    ),
                    float(
                        b["y"] - a["y"]
                    ),
                    float(
                        b["x"] - a["x"]
                    ),
                ])
                * scale
            )

            physical_distances.append(
                float(
                    np.linalg.norm(
                        delta_um
                    )
                )
            )

            sm = int(
                a["match_node_id"]
            )

            tm = int(
                b["match_node_id"]
            )

            if (
                sm >= 0
                and tm >= 0
                and (sm, tm)
                in gt_edge_set
            ):
                direct_true_edges += 1

        if physical_distances:

            median_motion_um = float(
                np.median(
                    physical_distances
                )
            )

            max_motion_um = float(
                np.max(
                    physical_distances
                )
            )

        else:

            median_motion_um = np.nan
            max_motion_um = np.nan

        # -----------------------------------------
        # GT diagnostic support
        # -----------------------------------------

        matched_nodes = int(
            (
                comp_nodes[
                    "match_node_id"
                ] >= 0
            ).sum()
        )

        contains_matched_node = (
            matched_nodes > 0
        )

        contains_true_edge = (
            direct_true_edges > 0
        )

        component_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "n_nodes":
                n_nodes,

            "n_edges":
                n_edges,

            "temporal_span":
                temporal_span,

            "density_in_span":
                n_nodes
                / max(
                    temporal_span,
                    1,
                ),

            "mean_edge_prob":
                mean_edge_prob,

            "median_edge_prob":
                median_edge_prob,

            "min_edge_prob":
                min_edge_prob,

            "median_motion_um":
                median_motion_um,

            "max_motion_um":
                max_motion_um,

            "matched_nodes":
                matched_nodes,

            "contains_matched_node":
                contains_matched_node,

            "direct_true_edges":
                direct_true_edges,

            "contains_true_edge":
                contains_true_edge,
        })

    if i % 10 == 0:
        print(
            f"Processed {i}/40"
        )


component_df = pd.DataFrame(
    component_rows
)

print(
    "\nComponents:",
    len(component_df),
)


# =========================================================
# Size buckets
# =========================================================

def size_bucket(n):

    if n == 1:
        return "1"

    if n == 2:
        return "2"

    if n <= 4:
        return "3-4"

    if n <= 9:
        return "5-9"

    if n <= 19:
        return "10-19"

    return "20+"


component_df[
    "size_bucket"
] = (
    component_df[
        "n_nodes"
    ].map(size_bucket)
)


print(
    "\n=== COMPONENT SIZE DISTRIBUTION ==="
)

display(
    component_df[
        "size_bucket"
    ]
    .value_counts()
    .reindex([
        "1",
        "2",
        "3-4",
        "5-9",
        "10-19",
        "20+",
    ])
    .to_frame(
        "components"
    )
)


print(
    "\n=== GT SUPPORT BY COMPONENT SIZE ==="
)

display(
    component_df
    .groupby(
        "size_bucket",
        observed=True,
    )
    .agg(
        components=(
            "n_nodes",
            "size",
        ),

        mean_nodes=(
            "n_nodes",
            "mean",
        ),

        matched_node_component_rate=(
            "contains_matched_node",
            "mean",
        ),

        true_edge_component_rate=(
            "contains_true_edge",
            "mean",
        ),

        mean_edge_prob=(
            "mean_edge_prob",
            "mean",
        ),

        median_edge_prob=(
            "median_edge_prob",
            "median",
        ),

        median_motion_um=(
            "median_motion_um",
            "median",
        ),
    )
)


print(
    "\n=== COMPONENT QUALITY: TRUE-EDGE SUPPORT ==="
)

display(
    component_df[
        component_df[
            "n_nodes"
        ] >= 2
    ]
    .groupby(
        "contains_true_edge"
    )[
        [
            "n_nodes",
            "temporal_span",
            "mean_edge_prob",
            "min_edge_prob",
            "median_motion_um",
            "max_motion_um",
        ]
    ]
    .agg([
        "count",
        "mean",
        "median",
    ])
)



component 越长，得到 GT support 的概率单调增加

平均 edge confidence 也有信号

In [ ]:
# =========================================================
# Conditional component-confidence audit — FIXED
#
# Control for:
#   1. prefix
#   2. component size
#
# Then test whether mean_edge_prob still provides
# independent enrichment for GT-supported components.
#
# Uses transform instead of groupby.apply so grouping
# columns are preserved.
# =========================================================

cond_df = (
    component_df[
        component_df["n_nodes"] >= 2
    ]
    .copy()
)

cond_df = (
    cond_df[
        cond_df["mean_edge_prob"].notna()
    ]
    .copy()
)

print(
    "Components entering conditional audit:",
    len(cond_df),
)

print(
    "Columns:",
    cond_df.columns.tolist(),
)

assert "prefix" in cond_df.columns
assert "size_bucket" in cond_df.columns


# =========================================================
# Within each prefix × size bucket:
# assign mean_edge_prob quartiles
# =========================================================

QUARTILE_LABELS = [
    "Q1_low",
    "Q2",
    "Q3",
    "Q4_high",
]


def quartile_with_ties_broken(series):
    """
    Rank first to break exact edge-probability ties,
    then assign four equal-frequency quartiles.
    """

    ranked = series.rank(
        method="first"
    )

    return pd.qcut(
        ranked,
        q=4,
        labels=QUARTILE_LABELS,
    )


cond_df[
    "confidence_quartile"
] = (
    cond_df
    .groupby(
        [
            "prefix",
            "size_bucket",
        ],
        observed=True,
    )[
        "mean_edge_prob"
    ]
    .transform(
        quartile_with_ties_broken
    )
)


print(
    "\nQuartile missing values:",
    cond_df[
        "confidence_quartile"
    ].isna().sum(),
)

assert (
    cond_df[
        "confidence_quartile"
    ].isna().sum()
    == 0
)


# =========================================================
# Main conditional table:
# prefix × size × confidence quartile
# =========================================================

conditional_summary = (
    cond_df
    .groupby(
        [
            "prefix",
            "size_bucket",
            "confidence_quartile",
        ],
        observed=True,
    )
    .agg(
        components=(
            "n_nodes",
            "size",
        ),

        median_n_nodes=(
            "n_nodes",
            "median",
        ),

        median_edge_prob=(
            "mean_edge_prob",
            "median",
        ),

        matched_node_component_rate=(
            "contains_matched_node",
            "mean",
        ),

        true_edge_component_rate=(
            "contains_true_edge",
            "mean",
        ),

        median_temporal_span=(
            "temporal_span",
            "median",
        ),
    )
    .reset_index()
)


print(
    "\n=== CONDITIONAL CONFIDENCE AUDIT ==="
)

display(
    conditional_summary
)


# =========================================================
# Collapse prefixes AFTER quartiles have already been
# assigned independently within prefix × size.
# =========================================================

collapsed = (
    cond_df
    .groupby(
        [
            "size_bucket",
            "confidence_quartile",
        ],
        observed=True,
    )
    .agg(
        components=(
            "n_nodes",
            "size",
        ),

        median_edge_prob=(
            "mean_edge_prob",
            "median",
        ),

        matched_node_component_rate=(
            "contains_matched_node",
            "mean",
        ),

        true_edge_component_rate=(
            "contains_true_edge",
            "mean",
        ),
    )
    .reset_index()
)


# Force sensible display ordering
size_order = [
    "2",
    "3-4",
    "5-9",
    "10-19",
    "20+",
]

quartile_order = [
    "Q1_low",
    "Q2",
    "Q3",
    "Q4_high",
]

collapsed[
    "size_bucket"
] = pd.Categorical(
    collapsed["size_bucket"],
    categories=size_order,
    ordered=True,
)

collapsed[
    "confidence_quartile"
] = pd.Categorical(
    collapsed["confidence_quartile"],
    categories=quartile_order,
    ordered=True,
)

collapsed = (
    collapsed
    .sort_values(
        [
            "size_bucket",
            "confidence_quartile",
        ]
    )
    .reset_index(drop=True)
)


print(
    "\n=== COLLAPSED CONDITIONAL TREND ==="
)

display(
    collapsed
)


# =========================================================
# Q4 vs Q1 enrichment separately by prefix × size
# =========================================================

enrichment_rows = []

for (
    prefix,
    bucket
), group in conditional_summary.groupby(
    [
        "prefix",
        "size_bucket",
    ],
    observed=True,
):

    lookup = (
        group
        .set_index(
            "confidence_quartile"
        )
    )

    if (
        "Q1_low" not in lookup.index
        or "Q4_high" not in lookup.index
    ):
        continue

    low_true = float(
        lookup.loc[
            "Q1_low",
            "true_edge_component_rate",
        ]
    )

    high_true = float(
        lookup.loc[
            "Q4_high",
            "true_edge_component_rate",
        ]
    )

    low_matched = float(
        lookup.loc[
            "Q1_low",
            "matched_node_component_rate",
        ]
    )

    high_matched = float(
        lookup.loc[
            "Q4_high",
            "matched_node_component_rate",
        ]
    )

    enrichment_rows.append({
        "prefix":
            prefix,

        "size_bucket":
            bucket,

        "Q1_true_edge_rate":
            low_true,

        "Q4_true_edge_rate":
            high_true,

        "true_edge_absolute_delta":
            high_true - low_true,

        "true_edge_Q4_over_Q1":
            (
                high_true / low_true
                if low_true > 0
                else np.nan
            ),

        "Q1_matched_node_rate":
            low_matched,

        "Q4_matched_node_rate":
            high_matched,

        "matched_node_absolute_delta":
            high_matched - low_matched,

        "matched_node_Q4_over_Q1":
            (
                high_matched / low_matched
                if low_matched > 0
                else np.nan
            ),
    })


enrichment_df = pd.DataFrame(
    enrichment_rows
)

enrichment_df[
    "size_bucket"
] = pd.Categorical(
    enrichment_df["size_bucket"],
    categories=size_order,
    ordered=True,
)

enrichment_df = (
    enrichment_df
    .sort_values(
        [
            "prefix",
            "size_bucket",
        ]
    )
    .reset_index(drop=True)
)


print(
    "\n=== Q4 vs Q1 GT-SUPPORT ENRICHMENT ==="
)

display(
    enrichment_df
)


# =========================================================
# Compact pivot for easiest interpretation
# =========================================================

print(
    "\n=== TRUE-EDGE RATE PIVOT ==="
)

true_edge_pivot = (
    collapsed
    .pivot(
        index="size_bucket",
        columns="confidence_quartile",
        values="true_edge_component_rate",
    )
)

display(
    true_edge_pivot
)


print(
    "\nConditional confidence audit: OK"
)

component persistence 有价值；component mean edge confidence 不是通用质量指标

detection/node quality 是主要瓶颈​

In [ ]:
# =========================================================
# Inspect prediction node attributes
# Goal: determine whether detector confidence was saved
# =========================================================

sample_path = sorted(
    PRED_DIR.glob("*.geff")
)[0]

sample_name = sample_path.stem

sample_g = load_graph(
    sample_path
)

node_attrs = (
    sample_g
    .node_attrs(unpack=True)
    .to_pandas()
)

edge_attrs = (
    sample_g
    .edge_attrs(unpack=True)
    .to_pandas()
)

print("Dataset:", sample_name)

print("\n=== NODE COLUMNS ===")
print(
    node_attrs.columns.tolist()
)

print("\n=== NODE SAMPLE ===")
display(
    node_attrs.head()
)

print("\n=== NODE DTYPES ===")
display(
    node_attrs.dtypes.to_frame(
        "dtype"
    )
)

print("\n=== NUMERIC NODE SUMMARY ===")

numeric_node_cols = (
    node_attrs
    .select_dtypes(
        include=[np.number]
    )
    .columns
)

display(
    node_attrs[
        numeric_node_cols
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.50,
            0.90,
            0.95,
            0.99,
        ]
    ).T
)

print("\n=== EDGE COLUMNS ===")
print(
    edge_attrs.columns.tolist()
)

In [ ]:
# =========================================================
# Read-only audit:
# Where is detection confidence produced and discarded?
# =========================================================

from pathlib import Path
import re

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

LOCAL_PREDICT = (
    PROJECT
    / "scripts"
    / "predict_unet_transformer_mps.py"
)

OFFICIAL_ROOTS = [
    PROJECT / "external" / "official" / "scripts",
    PROJECT / "external" / "official" / "src",
]

SEARCH_PATTERNS = [
    r"det_threshold",
    r"threshold",
    r"local.?max",
    r"peak",
    r"detection",
    r"prob",
    r"score",
    r"node_attrs",
    r"add_node",
    r"node_id",
    r"IndexedRXGraph",
    r"from_geff",
    r"to_geff",
]


def show_matches(path, patterns, context=4):
    text = path.read_text(
        encoding="utf-8"
    )

    lines = text.splitlines()

    hits = []

    for i, line in enumerate(lines):
        if any(
            re.search(
                pattern,
                line,
                flags=re.IGNORECASE,
            )
            for pattern in patterns
        ):
            hits.append(i)

    # merge nearby hits so output is not duplicated
    blocks = []

    for i in hits:
        start = max(
            0,
            i - context,
        )

        end = min(
            len(lines),
            i + context + 1,
        )

        if (
            blocks
            and start <= blocks[-1][1] + 1
        ):
            blocks[-1] = (
                blocks[-1][0],
                max(
                    blocks[-1][1],
                    end,
                ),
            )
        else:
            blocks.append(
                (start, end)
            )

    print(
        "\n" + "=" * 80
    )
    print(path)
    print("=" * 80)

    if not blocks:
        print("No relevant matches.")
        return

    for start, end in blocks:

        print(
            f"\n--- lines {start + 1}-{end} ---"
        )

        for j in range(
            start,
            end,
        ):
            print(
                f"{j + 1:5d}: "
                f"{lines[j]}"
            )


# =========================================================
# 1. Local MPS inference adapter
# =========================================================

assert LOCAL_PREDICT.exists()

show_matches(
    LOCAL_PREDICT,
    SEARCH_PATTERNS,
    context=5,
)


# =========================================================
# 2. Find likely official prediction/detection files
# =========================================================

candidate_files = []

for root in OFFICIAL_ROOTS:

    for path in root.rglob("*.py"):

        text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        score = sum(
            bool(
                re.search(
                    pattern,
                    text,
                    flags=re.IGNORECASE,
                )
            )
            for pattern in [
                r"det_threshold",
                r"local.?max",
                r"detection",
                r"edge_prob",
            ]
        )

        if score >= 2:
            candidate_files.append(
                (
                    score,
                    path,
                )
            )


candidate_files = sorted(
    candidate_files,
    key=lambda x: (
        -x[0],
        str(x[1]),
    ),
)


print(
    "\n\n=== LIKELY OFFICIAL FILES ==="
)

for score, path in candidate_files[:15]:
    print(
        f"[score={score}] "
        f"{path.relative_to(PROJECT)}"
    )


# =========================================================
# 3. Show snippets from top official candidates
# =========================================================

for score, path in candidate_files[:5]:

    show_matches(
        path,
        SEARCH_PATTERNS,
        context=5,
    )

In [ ]:
# =========================================================
# Targeted audit of detector confidence flow
# =========================================================

from pathlib import Path
import re

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PREDICT_FILE = (
    PROJECT
    / "scripts"
    / "predict_unet_transformer_mps.py"
)

assert PREDICT_FILE.exists()

text = PREDICT_FILE.read_text(
    encoding="utf-8"
)

lines = text.splitlines()


def show_context(hit_lines, before=12, after=20):

    blocks = []

    for idx in hit_lines:

        start = max(
            0,
            idx - before,
        )

        end = min(
            len(lines),
            idx + after + 1,
        )

        if (
            blocks
            and start <= blocks[-1][1]
        ):
            blocks[-1] = (
                blocks[-1][0],
                max(
                    blocks[-1][1],
                    end,
                ),
            )
        else:
            blocks.append(
                (start, end)
            )

    for start, end in blocks:

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"LINES {start + 1}-{end}"
        )

        print(
            "=" * 80
        )

        for j in range(
            start,
            end,
        ):
            print(
                f"{j + 1:5d}: {lines[j]}"
            )


# =========================================================
# 1. Every actual det_threshold use
# =========================================================

det_hits = [
    i
    for i, line in enumerate(lines)
    if "det_threshold" in line
]

print(
    "det_threshold occurrences:",
    [
        i + 1
        for i in det_hits
    ],
)

show_context(
    det_hits,
    before=15,
    after=30,
)


# =========================================================
# 2. Likely detector-probability production
# =========================================================

patterns = [
    r"sigmoid",
    r"local.?max",
    r"max.?pool",
    r"peak",
    r"heatmap",
    r"logit",
    r"prob",
]

prob_hits = []

for i, line in enumerate(lines):

    if any(
        re.search(
            p,
            line,
            flags=re.IGNORECASE,
        )
        for p in patterns
    ):
        prob_hits.append(i)


print(
    "\nProbability/local-max related occurrences:",
    len(prob_hits),
)

show_context(
    prob_hits,
    before=8,
    after=12,
)


# =========================================================
# 3. Likely graph-node construction / serialization
# =========================================================

graph_patterns = [
    r"add_node",
    r"add_nodes",
    r"node_attrs",
    r"IndexedRXGraph",
    r"to_geff",
    r"save",
]

graph_hits = []

for i, line in enumerate(lines):

    if any(
        re.search(
            p,
            line,
            flags=re.IGNORECASE,
        )
        for p in graph_patterns
    ):
        graph_hits.append(i)


print(
    "\nGraph/write related occurrences:",
    len(graph_hits),
)

show_context(
    graph_hits,
    before=10,
    after=20,
)

In [ ]:
# =========================================================
# Print only the actual detector-confidence code paths
# =========================================================

from pathlib import Path

PREDICT_FILE = (
    Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
    / "scripts"
    / "predict_unet_transformer_mps.py"
)

lines = PREDICT_FILE.read_text(
    encoding="utf-8"
).splitlines()


def print_range(start, end):
    print("\n" + "=" * 80)
    print(f"LINES {start}-{end}")
    print("=" * 80)

    for i in range(start - 1, end):
        print(f"{i + 1:5d}: {lines[i]}")


# det_threshold actual runtime uses
print_range(250, 325)

# likely graph/node construction
print_range(390, 435)

# CLI definition only, for confirmation
print_range(665, 690)

In [ ]:
# =========================================================
# Precise detector-confidence audit
# =========================================================

from pathlib import Path

PREDICT_FILE = (
    Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
    / "scripts"
    / "predict_unet_transformer_mps.py"
)

lines = PREDICT_FILE.read_text(
    encoding="utf-8"
).splitlines()


def show_around(line_no, radius=8):
    start = max(1, line_no - radius)
    end = min(len(lines), line_no + radius)

    print("\n" + "=" * 70)
    print(f"AROUND LINE {line_no}")
    print("=" * 70)

    for n in range(start, end + 1):
        print(f"{n:5d}: {lines[n - 1]}")


for line_no in [
    276,
    290,
    304,
    415,
]:
    show_around(
        line_no,
        radius=8,
    )

In [ ]:
# =========================================================
# Extract the complete _detect_cells_pooled function
# without importing/running the prediction script
# =========================================================

from pathlib import Path
import ast

PREDICT_FILE = (
    Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
    / "scripts"
    / "predict_unet_transformer_mps.py"
)

source = PREDICT_FILE.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)

target_name = "_detect_cells_pooled"

for node in tree.body:
    if (
        isinstance(node, ast.FunctionDef)
        and node.name == target_name
    ):
        function_source = ast.get_source_segment(
            source,
            node,
        )

        print(function_source)
        break
else:
    raise RuntimeError(
        f"{target_name} not found"
    )

In [ ]:
# =========================================================
# Compact detector implementation audit
# Print only lines relevant to probability / peak selection
# =========================================================

from pathlib import Path
import ast

PREDICT_FILE = (
    Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
    / "scripts"
    / "predict_unet_transformer_mps.py"
)

source = PREDICT_FILE.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)

for node in tree.body:
    if (
        isinstance(node, ast.FunctionDef)
        and node.name == "_detect_cells_pooled"
    ):
        func_lines = ast.get_source_segment(
            source,
            node,
        ).splitlines()
        break
else:
    raise RuntimeError(
        "_detect_cells_pooled not found"
    )


KEYWORDS = [
    "sigmoid",
    "pool",
    "threshold",
    "peak",
    "nonzero",
    "where",
    "return",
]

print(
    "=== DETECTOR CORE LINES ==="
)

for i, line in enumerate(
    func_lines,
    start=1,
):
    if any(
        keyword in line.lower()
        for keyword in KEYWORDS
    ):
        start = max(
            0,
            i - 3,
        )
        end = min(
            len(func_lines),
            i + 2,
        )

        print(
            f"\n--- around function line {i} ---"
        )

        for j in range(
            start,
            end,
        ):
            print(
                f"{j + 1:3d}: "
                f"{func_lines[j]}"
            )

In [ ]:
# =========================================================
# FINAL plumbing check:
# show the exact call sites of _detect_cells_pooled
# =========================================================

from pathlib import Path
import ast

PREDICT_FILE = (
    Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
    / "scripts"
    / "predict_unet_transformer_mps.py"
)

source = PREDICT_FILE.read_text(
    encoding="utf-8"
)

lines = source.splitlines()

tree = ast.parse(source)

call_lines = []

for node in ast.walk(tree):
    if (
        isinstance(node, ast.Call)
        and isinstance(node.func, ast.Name)
        and node.func.id == "_detect_cells_pooled"
    ):
        call_lines.append(
            node.lineno
        )

print(
    "_detect_cells_pooled call lines:",
    call_lines,
)

for line_no in call_lines:

    start = max(
        1,
        line_no - 12,
    )

    end = min(
        len(lines),
        line_no + 18,
    )

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"CALL AROUND LINE {line_no}"
    )

    print(
        "=" * 72
    )

    print(
        "\n".join(
            f"{n:5d}: {lines[n - 1]}"
            for n in range(
                start,
                end + 1,
            )
        )
    )

In [ ]:
# =========================================================
# Coarse detection-confidence audit from threshold survival
#
# Existing inference runs:
#   0.9900
#   0.9925
#   0.9950
#   0.9975
#
# No new inference required.
# =========================================================

from collections import defaultdict

DET_DIRS = {
    0.9900: (
        PROJECT
        / "predictions"
        / "markdd520"
        / "official_baseline_fold0"
        / "split_0"
    ),

    0.9925: (
        PROJECT
        / "predictions"
        / "markdd520"
        / "official_baseline_fold0_det09925"
        / "split_0"
    ),

    0.9950: (
        PROJECT
        / "predictions"
        / "markdd520"
        / "official_baseline_fold0_det09950"
        / "split_0"
    ),

    0.9975: (
        PROJECT
        / "predictions"
        / "markdd520"
        / "official_baseline_fold0_det09975"
        / "split_0"
    ),
}


# =========================================================
# Coordinate key
#
# Node IDs cannot be compared between threshold runs.
# Detection coordinates can.
# =========================================================

def node_coordinate_keys(graph):

    df = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
    )

    keys = {
        (
            int(row.t),
            int(round(row.z)),
            int(round(row.y)),
            int(round(row.x)),
        )
        for row in df.itertuples(
            index=False
        )
    }

    return keys


# =========================================================
# First verify that thresholded detections are truly nested
# =========================================================

nesting_rows = []

dataset_key_sets = {}

for name in sorted(
    fold0["dataset"]
):

    threshold_sets = {}

    for threshold, pred_dir in DET_DIRS.items():

        path = (
            pred_dir
            / f"{name}.geff"
        )

        assert path.exists(), path

        g = load_graph(path)

        threshold_sets[
            threshold
        ] = node_coordinate_keys(g)

    dataset_key_sets[name] = threshold_sets

    thresholds = [
        0.9900,
        0.9925,
        0.9950,
        0.9975,
    ]

    for low, high in zip(
        thresholds[:-1],
        thresholds[1:],
    ):

        low_set = threshold_sets[low]
        high_set = threshold_sets[high]

        violations = (
            high_set
            - low_set
        )

        nesting_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "low_threshold":
                low,

            "high_threshold":
                high,

            "n_low":
                len(low_set),

            "n_high":
                len(high_set),

            "high_not_in_low":
                len(violations),
        })


nesting_df = pd.DataFrame(
    nesting_rows
)

print(
    "=== THRESHOLD NESTING CHECK ==="
)

display(
    nesting_df
    .groupby(
        [
            "low_threshold",
            "high_threshold",
        ]
    )
    .agg(
        datasets=(
            "dataset",
            "size",
        ),

        low_nodes=(
            "n_low",
            "sum",
        ),

        high_nodes=(
            "n_high",
            "sum",
        ),

        violations=(
            "high_not_in_low",
            "sum",
        ),
    )
    .reset_index()
)

total_violations = int(
    nesting_df[
        "high_not_in_low"
    ].sum()
)

print(
    "\nTotal nesting violations:",
    total_violations,
)


# =========================================================
# Use 0.99 nodes as the candidate universe.
#
# Official matching supplies diagnostic GT support only.
# =========================================================

confidence_rows = []

BASE_THRESHOLD = 0.9900
BASE_DIR = DET_DIRS[
    BASE_THRESHOLD
]


for i, name in enumerate(
    sorted(
        fold0["dataset"]
    ),
    start=1,
):

    pred_g = load_graph(
        BASE_DIR
        / f"{name}.geff"
    )

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    scale = np.asarray(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    # Diagnostic node matching
    _ = official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(scale),
        max_distance=7.0,
    )

    nodes = (
        pred_g
        .node_attrs(unpack=True)
        .to_pandas()
    )

    sets = dataset_key_sets[
        name
    ]

    set_09925 = sets[
        0.9925
    ]

    set_09950 = sets[
        0.9950
    ]

    set_09975 = sets[
        0.9975
    ]

    # Optional duplicate-coordinate sanity check
    base_keys = node_coordinate_keys(
        pred_g
    )

    assert (
        len(base_keys)
        == pred_g.num_nodes()
    ), (
        f"Duplicate detection coordinates "
        f"in {name}"
    )

    for row in nodes.itertuples(
        index=False
    ):

        key = (
            int(row.t),
            int(round(row.z)),
            int(round(row.y)),
            int(round(row.x)),
        )

        # Highest threshold survived
        if key in set_09975:

            confidence_bin = (
                ">=0.9975"
            )

            confidence_rank = 4

        elif key in set_09950:

            confidence_bin = (
                "[0.9950,0.9975)"
            )

            confidence_rank = 3

        elif key in set_09925:

            confidence_bin = (
                "[0.9925,0.9950)"
            )

            confidence_rank = 2

        else:

            confidence_bin = (
                "[0.9900,0.9925)"
            )

            confidence_rank = 1

        matched = (
            int(row.match_node_id)
            >= 0
        )

        confidence_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "t":
                int(row.t),

            "z":
                int(round(row.z)),

            "y":
                int(round(row.y)),

            "x":
                int(round(row.x)),

            "confidence_bin":
                confidence_bin,

            "confidence_rank":
                confidence_rank,

            "matched_gt_node":
                matched,
        })

    if i % 10 == 0:
        print(
            f"Processed {i}/40"
        )


det_conf_df = pd.DataFrame(
    confidence_rows
)


# =========================================================
# Ordering
# =========================================================

CONF_ORDER = [
    "[0.9900,0.9925)",
    "[0.9925,0.9950)",
    "[0.9950,0.9975)",
    ">=0.9975",
]

det_conf_df[
    "confidence_bin"
] = pd.Categorical(
    det_conf_df[
        "confidence_bin"
    ],
    categories=CONF_ORDER,
    ordered=True,
)


# =========================================================
# Overall
# =========================================================

overall_conf = (
    det_conf_df
    .groupby(
        "confidence_bin",
        observed=True,
    )
    .agg(
        nodes=(
            "matched_gt_node",
            "size",
        ),

        matched_nodes=(
            "matched_gt_node",
            "sum",
        ),

        matched_rate=(
            "matched_gt_node",
            "mean",
        ),
    )
    .reset_index()
)

print(
    "\n=== DETECTION CONFIDENCE vs GT MATCH ==="
)

display(
    overall_conf
)


# =========================================================
# Prefix-controlled result
# =========================================================

prefix_conf = (
    det_conf_df
    .groupby(
        [
            "prefix",
            "confidence_bin",
        ],
        observed=True,
    )
    .agg(
        nodes=(
            "matched_gt_node",
            "size",
        ),

        matched_nodes=(
            "matched_gt_node",
            "sum",
        ),

        matched_rate=(
            "matched_gt_node",
            "mean",
        ),
    )
    .reset_index()
)

print(
    "\n=== DETECTION CONFIDENCE vs GT MATCH BY PREFIX ==="
)

display(
    prefix_conf
)


# =========================================================
# Easier-to-read pivot
# =========================================================

print(
    "\n=== MATCH RATE PIVOT ==="
)

display(
    prefix_conf.pivot(
        index="prefix",
        columns="confidence_bin",
        values="matched_rate",
    )
)

print(
    "\nCoarse detection-confidence audit: OK"
)

In [ ]:
# =========================================================
# Detection confidence × component persistence audit
#
# Current baseline universe:
#   det_threshold = 0.995
#
# Compare:
#   low  = [0.9950, 0.9975)
#   high = >= 0.9975
#
# Components are computed AFTER frozen G2,
# but BEFORE component cleanup.
# =========================================================

node_context_rows = []

PRED_0995_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0_det09950"
    / "split_0"
)


def component_size_bucket(n):
    if n == 1:
        return "1"
    elif n == 2:
        return "2"
    elif n <= 4:
        return "3-4"
    elif n <= 9:
        return "5-9"
    elif n <= 19:
        return "10-19"
    else:
        return "20+"


for i, name in enumerate(
    sorted(fold0["dataset"]),
    start=1,
):

    pred_g = load_graph(
        PRED_0995_DIR
        / f"{name}.geff"
    )

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    scale = np.asarray(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    # =====================================================
    # Frozen G2
    # =====================================================

    apply_frozen_g2(
        pred_g,
        scale,
    )

    # =====================================================
    # Diagnostic official node matching
    # =====================================================

    _ = official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(scale),
        max_distance=7.0,
    )

    nodes = (
        pred_g
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    # =====================================================
    # Component membership
    # =====================================================

    components = component_sets(
        pred_g
    )

    node_to_component_size = {}

    for comp in components:

        comp_size = len(comp)

        for node_id in comp:
            node_to_component_size[
                int(node_id)
            ] = comp_size

    assert (
        len(node_to_component_size)
        == pred_g.num_nodes()
    )

    # =====================================================
    # Detection confidence from threshold survival
    # =====================================================

    high_conf_set = (
        dataset_key_sets[name][
            0.9975
        ]
    )

    for node_id, row in nodes.iterrows():

        key = (
            int(row["t"]),
            int(round(row["z"])),
            int(round(row["y"])),
            int(round(row["x"])),
        )

        survives_09975 = (
            key in high_conf_set
        )

        comp_size = (
            node_to_component_size[
                int(node_id)
            ]
        )

        node_context_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "node_id":
                int(node_id),

            "component_size":
                comp_size,

            "size_bucket":
                component_size_bucket(
                    comp_size
                ),

            "confidence_band":
                (
                    "high_>=0.9975"
                    if survives_09975
                    else "low_[0.995,0.9975)"
                ),

            "matched_gt_node":
                int(
                    row[
                        "match_node_id"
                    ]
                ) >= 0,
        })

    if i % 10 == 0:
        print(
            f"Processed {i}/40"
        )


node_context_df = pd.DataFrame(
    node_context_rows
)

print(
    "\nNodes analysed:",
    len(node_context_df),
)

assert (
    len(node_context_df)
    == 1206416
)


# =========================================================
# Main conditional table
# =========================================================

SIZE_ORDER = [
    "1",
    "2",
    "3-4",
    "5-9",
    "10-19",
    "20+",
]

context_summary = (
    node_context_df
    .groupby(
        [
            "prefix",
            "size_bucket",
            "confidence_band",
        ],
        observed=True,
    )
    .agg(
        nodes=(
            "matched_gt_node",
            "size",
        ),

        matched_nodes=(
            "matched_gt_node",
            "sum",
        ),

        matched_rate=(
            "matched_gt_node",
            "mean",
        ),
    )
    .reset_index()
)

context_summary[
    "size_bucket"
] = pd.Categorical(
    context_summary[
        "size_bucket"
    ],
    categories=SIZE_ORDER,
    ordered=True,
)

context_summary = (
    context_summary
    .sort_values(
        [
            "prefix",
            "size_bucket",
            "confidence_band",
        ]
    )
    .reset_index(drop=True)
)

print(
    "\n=== DET CONFIDENCE × COMPONENT SIZE ==="
)

display(
    context_summary
)


# =========================================================
# Easier comparison: low vs high confidence
# =========================================================

match_pivot = (
    context_summary
    .pivot(
        index=[
            "prefix",
            "size_bucket",
        ],
        columns="confidence_band",
        values="matched_rate",
    )
    .reset_index()
)

print(
    "\n=== MATCH RATE PIVOT ==="
)

display(
    match_pivot
)


# =========================================================
# High-confidence enrichment within each component size
# =========================================================

enrichment_rows = []

for (
    prefix,
    size_bucket
), group in context_summary.groupby(
    [
        "prefix",
        "size_bucket",
    ],
    observed=True,
):

    lookup = (
        group
        .set_index(
            "confidence_band"
        )
    )

    low_name = (
        "low_[0.995,0.9975)"
    )

    high_name = (
        "high_>=0.9975"
    )

    if (
        low_name not in lookup.index
        or high_name not in lookup.index
    ):
        continue

    low_rate = float(
        lookup.loc[
            low_name,
            "matched_rate",
        ]
    )

    high_rate = float(
        lookup.loc[
            high_name,
            "matched_rate",
        ]
    )

    enrichment_rows.append({
        "prefix":
            prefix,

        "size_bucket":
            size_bucket,

        "low_conf_match_rate":
            low_rate,

        "high_conf_match_rate":
            high_rate,

        "absolute_delta":
            high_rate - low_rate,

        "high_over_low":
            (
                high_rate / low_rate
                if low_rate > 0
                else np.nan
            ),
    })


context_enrichment = pd.DataFrame(
    enrichment_rows
)

context_enrichment[
    "size_bucket"
] = pd.Categorical(
    context_enrichment[
        "size_bucket"
    ],
    categories=SIZE_ORDER,
    ordered=True,
)

context_enrichment = (
    context_enrichment
    .sort_values(
        [
            "prefix",
            "size_bucket",
        ]
    )
    .reset_index(drop=True)
)

print(
    "\n=== CONFIDENCE ENRICHMENT WITHIN COMPONENT SIZE ==="
)

display(
    context_enrichment
)

print(
    "\nDetection confidence × component audit: OK"
)

即使控制 component size，>=0.9975 的节点仍然显著更容易得到 GT support

In [ ]:
# =========================================================
# Confidence-aware pruning official metric ablation
#
# Base prediction:
#   det = 0.995
#
# Low-confidence node:
#   present at 0.995
#   absent at 0.9975
#
# Policy:
#   remove low-confidence nodes ONLY when their
#   pre-pruning G2 component size <= K.
#
# Then apply the same frozen component cleanup:
#   - remove all singleton components
#   - remove 2-node components with displacement > 3.5 µm
# =========================================================

from collections import defaultdict

PRED_0995_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0_det09950"
    / "split_0"
)

CONF_COMPONENT_LIMITS = [
    0,      # baseline: no confidence pruning
    2,
    4,
    9,
    19,
    np.inf, # remove ALL low-confidence nodes
]


# =========================================================
# Frozen cleanup helper
# Self-contained for notebook 07
# =========================================================

def apply_frozen_component_cleanup(
    graph,
    scale,
):
    scale = np.asarray(
        scale,
        dtype=float,
    )

    nodes = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        graph
        .edge_attrs(
            attr_keys=["edge_prob"],
            unpack=True,
        )
        .to_pandas()
    )

    edge_lookup = {}

    for _, e in edges.iterrows():
        s = int(e["source_id"])
        t = int(e["target_id"])

        edge_lookup[
            frozenset([s, t])
        ] = (s, t)

    comps = component_sets(
        graph
    )

    remove_nodes = set()

    singleton_components = 0
    long_two_node_components = 0

    for comp in comps:

        comp = list(
            map(int, comp)
        )

        # -----------------------------------------
        # Singleton
        # -----------------------------------------

        if len(comp) == 1:

            remove_nodes.add(
                comp[0]
            )

            singleton_components += 1

        # -----------------------------------------
        # 2-node component
        # -----------------------------------------

        elif len(comp) == 2:

            key = frozenset(comp)

            if key not in edge_lookup:
                continue

            source, target = (
                edge_lookup[key]
            )

            a = nodes.loc[source]
            b = nodes.loc[target]

            delta_um = (
                np.array([
                    float(
                        b["z"] - a["z"]
                    ),
                    float(
                        b["y"] - a["y"]
                    ),
                    float(
                        b["x"] - a["x"]
                    ),
                ])
                * scale
            )

            dist_um = float(
                np.linalg.norm(
                    delta_um
                )
            )

            if (
                dist_um
                > TWO_NODE_MAX_DIST_UM
            ):

                remove_nodes.update(
                    comp
                )

                long_two_node_components += 1

    if remove_nodes:
        graph.bulk_remove_nodes(
            sorted(remove_nodes)
        )

    return {
        "nodes_removed":
            len(remove_nodes),

        "singleton_components":
            singleton_components,

        "long_two_node_components":
            long_two_node_components,
    }


# =========================================================
# Confidence pruning helper
# =========================================================

def apply_low_conf_component_pruning(
    graph,
    high_conf_coordinate_set,
    max_component_size,
):
    """
    Remove low-confidence nodes iff their component size
    BEFORE pruning is <= max_component_size.

    low confidence means:
      node exists at det=0.995
      but not at det=0.9975
    """

    if max_component_size == 0:
        return {
            "low_conf_nodes_removed": 0,
            "affected_components": 0,
        }

    nodes = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    comps = component_sets(
        graph
    )

    remove_nodes = set()
    affected_components = 0

    for comp in comps:

        comp = list(
            map(int, comp)
        )

        comp_size = len(comp)

        if (
            not np.isinf(
                max_component_size
            )
            and comp_size
            > max_component_size
        ):
            continue

        component_remove = []

        for node_id in comp:

            row = nodes.loc[
                node_id
            ]

            key = (
                int(row["t"]),
                int(round(row["z"])),
                int(round(row["y"])),
                int(round(row["x"])),
            )

            is_high_conf = (
                key
                in high_conf_coordinate_set
            )

            if not is_high_conf:
                component_remove.append(
                    node_id
                )

        if component_remove:

            affected_components += 1

            remove_nodes.update(
                component_remove
            )

    if remove_nodes:
        graph.bulk_remove_nodes(
            sorted(remove_nodes)
        )

    return {
        "low_conf_nodes_removed":
            len(remove_nodes),

        "affected_components":
            affected_components,
    }


# =========================================================
# Evaluate sweep
# =========================================================

all_results = []

for max_comp_size in (
    CONF_COMPONENT_LIMITS
):

    print(
        "\n"
        + "=" * 72
    )

    print(
        "Confidence pruning max component size:",
        max_comp_size,
    )

    print(
        "=" * 72
    )

    records = []

    total_raw_nodes = 0
    total_conf_removed = 0
    total_cleanup_removed = 0
    total_final_nodes = 0
    total_affected_components = 0

    for i, name in enumerate(
        sorted(fold0["dataset"]),
        start=1,
    ):

        pred_g = load_graph(
            PRED_0995_DIR
            / f"{name}.geff"
        )

        gt_g = load_graph(
            TRAIN_DIR
            / f"{name}.geff"
        )

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        total_raw_nodes += (
            pred_g.num_nodes()
        )

        # -----------------------------------------
        # 1. Frozen G2
        # -----------------------------------------

        apply_frozen_g2(
            pred_g,
            scale,
        )

        # -----------------------------------------
        # 2. Confidence-aware pruning
        # -----------------------------------------

        high_conf_set = (
            dataset_key_sets[
                name
            ][0.9975]
        )

        conf_stats = (
            apply_low_conf_component_pruning(
                pred_g,
                high_conf_set,
                max_comp_size,
            )
        )

        total_conf_removed += (
            conf_stats[
                "low_conf_nodes_removed"
            ]
        )

        total_affected_components += (
            conf_stats[
                "affected_components"
            ]
        )

        # -----------------------------------------
        # 3. Same frozen component cleanup
        # -----------------------------------------

        before_cleanup = (
            pred_g.num_nodes()
        )

        cleanup_stats = (
            apply_frozen_component_cleanup(
                pred_g,
                scale,
            )
        )

        after_cleanup = (
            pred_g.num_nodes()
        )

        total_cleanup_removed += (
            before_cleanup
            - after_cleanup
        )

        total_final_nodes += (
            after_cleanup
        )

        # -----------------------------------------
        # 4. Locked official evaluation
        # -----------------------------------------

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(
                estimated_map[name]
            ),
            recall,
        )

        records.append(m)

        if i % 10 == 0:
            print(
                f"Processed {i}/40"
            )

    s = summarise(
        records
    )

    all_results.append({
        "max_component_size":
            max_comp_size,

        "confidence_nodes_removed":
            total_conf_removed,

        "affected_components":
            total_affected_components,

        "cleanup_nodes_removed":
            total_cleanup_removed,

        "final_pred_nodes":
            total_final_nodes,

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "division_tp":
            s["division_tp"],

        "division_fp":
            s["division_fp"],

        "division_fn":
            s["division_fn"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    })


confidence_pruning_results = (
    pd.DataFrame(
        all_results
    )
)

print(
    "\n=== CONFIDENCE-AWARE PRUNING SWEEP ==="
)

display(
    confidence_pruning_results
)


# =========================================================
# Baseline reproduction check
# =========================================================

baseline_row = (
    confidence_pruning_results[
        confidence_pruning_results[
            "max_component_size"
        ] == 0
    ]
    .iloc[0]
)

print(
    "\nBaseline reproduction:"
)

print(
    "score =",
    baseline_row["score"],
)

print(
    "difference vs frozen =",
    baseline_row["score"]
    - 0.741854,
)

print(
    "\nConfidence-aware pruning sweep: OK"
)

low-confidence pruning for components ≤4​

In [ ]:
# =========================================================
# Robustness check:
# baseline vs confidence-aware K<=4
#
# Evaluate:
#   - overall
#   - by prefix
#   - paired per-dataset changes
# =========================================================

COMPARE_LIMITS = {
    "baseline": 0,
    "conf_K4": 4,
}

comparison_records = {}
comparison_dataset_rows = []


for variant, max_comp_size in COMPARE_LIMITS.items():

    print(
        "\n" + "=" * 72
    )
    print(
        f"Evaluating {variant} | "
        f"max_component_size={max_comp_size}"
    )
    print(
        "=" * 72
    )

    records = []

    for i, name in enumerate(
        sorted(fold0["dataset"]),
        start=1,
    ):

        pred_g = load_graph(
            PRED_0995_DIR
            / f"{name}.geff"
        )

        gt_g = load_graph(
            TRAIN_DIR
            / f"{name}.geff"
        )

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        # -----------------------------------------
        # 1. Frozen G2
        # -----------------------------------------

        apply_frozen_g2(
            pred_g,
            scale,
        )

        # -----------------------------------------
        # 2. Confidence-aware pruning
        # -----------------------------------------

        conf_stats = (
            apply_low_conf_component_pruning(
                pred_g,
                dataset_key_sets[name][0.9975],
                max_comp_size,
            )
        )

        # -----------------------------------------
        # 3. Frozen component cleanup
        # -----------------------------------------

        apply_frozen_component_cleanup(
            pred_g,
            scale,
        )

        # -----------------------------------------
        # 4. Official metric
        # -----------------------------------------

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(
                estimated_map[name]
            ),
            recall,
        )

        records.append(m)

        comparison_dataset_rows.append({
            "variant":
                variant,

            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "confidence_nodes_removed":
                conf_stats[
                    "low_conf_nodes_removed"
                ],

            "edge_jaccard":
                m["edge_jaccard"],

            "adj_edge_jaccard":
                m["adj_edge_jaccard"],

            "node_recall":
                m["node_recall"],

            "node_ratio":
                m["total_node_ratio"],
        })

        if i % 10 == 0:
            print(
                f"{variant}: {i}/40"
            )

    comparison_records[
        variant
    ] = records


comparison_df = pd.DataFrame(
    comparison_dataset_rows
)


# =========================================================
# Overall sanity check
# =========================================================

overall_rows = []

for variant, records in comparison_records.items():

    s = summarise(records)

    overall_rows.append({
        "variant":
            variant,

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    })


overall_compare = pd.DataFrame(
    overall_rows
)

print(
    "\n=== OVERALL REPRODUCTION ==="
)

display(
    overall_compare
)


# =========================================================
# Prefix summaries
# =========================================================

prefix_rows = []

names = sorted(
    fold0["dataset"]
)

for variant, records in comparison_records.items():

    for prefix in [
        "44b6",
        "6bba",
    ]:

        prefix_records = [
            m
            for name, m in zip(
                names,
                records,
            )
            if prefix_map[name] == prefix
        ]

        s = summarise(
            prefix_records
        )

        ratios = [
            m["total_node_ratio"]
            for name, m in zip(
                names,
                records,
            )
            if prefix_map[name] == prefix
        ]

        prefix_rows.append({
            "variant":
                variant,

            "prefix":
                prefix,

            "n":
                s["n"],

            "edge_jaccard":
                s["edge_jaccard"],

            "adj_edge_jaccard":
                s["adj_edge_jaccard"],

            "division_jaccard":
                s["division_jaccard"],

            "node_recall":
                s["node_recall"],

            "median_node_ratio":
                float(
                    np.median(ratios)
                ),

            "score":
                s["score"],
        })


prefix_compare = (
    pd.DataFrame(
        prefix_rows
    )
    .sort_values(
        [
            "prefix",
            "variant",
        ]
    )
)

print(
    "\n=== BASELINE vs CONF-K4 BY PREFIX ==="
)

display(
    prefix_compare
)


# =========================================================
# Paired dataset-level change
# =========================================================

wide = (
    comparison_df
    .pivot(
        index=[
            "dataset",
            "prefix",
        ],
        columns="variant",
        values=[
            "edge_jaccard",
            "adj_edge_jaccard",
            "node_recall",
            "node_ratio",
        ],
    )
)

paired = pd.DataFrame(
    index=wide.index
)

for metric in [
    "edge_jaccard",
    "adj_edge_jaccard",
    "node_recall",
    "node_ratio",
]:

    paired[
        f"delta_{metric}"
    ] = (
        wide[
            (metric, "conf_K4")
        ]
        - wide[
            (metric, "baseline")
        ]
    )

paired = (
    paired
    .reset_index()
)


print(
    "\n=== PAIRED DATASET DELTAS ==="
)

display(
    paired.groupby(
        "prefix"
    ).agg(
        datasets=(
            "dataset",
            "size",
        ),

        mean_delta_edgeJ=(
            "delta_edge_jaccard",
            "mean",
        ),

        median_delta_edgeJ=(
            "delta_edge_jaccard",
            "median",
        ),

        edgeJ_improved_rate=(
            "delta_edge_jaccard",
            lambda x: float(
                (x > 0).mean()
            ),
        ),

        mean_delta_adjEdgeJ=(
            "delta_adj_edge_jaccard",
            "mean",
        ),

        adjEdgeJ_improved_rate=(
            "delta_adj_edge_jaccard",
            lambda x: float(
                (x > 0).mean()
            ),
        ),

        mean_delta_recall=(
            "delta_node_recall",
            "mean",
        ),

        mean_delta_node_ratio=(
            "delta_node_ratio",
            "mean",
        ),
    )
)


print(
    "\nBaseline vs confidence-K4 robustness check: OK"
)

baseline-v4 score=0.744252
	​


相对之前：

0.741854→0.744252

即：

+0.002398
	​


Detector
    det_threshold = 0.995
    pool_kernel_um = 3.0



Association
    official transformer



Division

    G2:
        p2 >= 0.55
        daughter separation >= 6 µm
        daughter angle >= 120°
        max parent-daughter distance <= 8 µm
        distance balance <= 0.50



Confidence-aware cleanup
    det_prob < 0.9975
    AND pre-pruning component size <= 4

        → remove node



Graph cleanup

    remove singleton components

    remove 2-node components with physical displacement > 3.5 µm

In [ ]:
# =========================================================
# PATCH — Restore inference script before pool-kernel work
#
# Purpose:
#   Revert the temporary pool-kernel CLI modification and
#   return to the last clean baseline-v4 state.
#
# Keep after running:
#   NO — DELETE THIS CELL after successful restoration.
# =========================================================

from pathlib import Path
import shutil
import hashlib

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PREDICT_FILE = (
    PROJECT
    / "scripts/predict_unet_transformer_mps.py"
)

BACKUP_FILE = (
    PROJECT
    / "scripts/predict_unet_transformer_mps.py.pre_pool_cli_backup"
)

assert BACKUP_FILE.exists()

shutil.copy2(
    BACKUP_FILE,
    PREDICT_FILE,
)

def sha256(path):
    h = hashlib.sha256()
    h.update(path.read_bytes())
    return h.hexdigest()

restored_sha = sha256(PREDICT_FILE)

print("Restored SHA256:", restored_sha)

assert restored_sha == (
    "23b76f7c90ffe698e955efee2b4333277a07f5b58b7eff6e580f2bb2ff49bc55"
)

print("Inference script restored: OK")

In [ ]:
# =========================================================
# SETUP — 07.09 Realized Pool Kernel Audit
#
# Purpose:
#   Determine the ACTUAL discrete max-pooling kernels
#   produced by different pool_kernel_um values for the
#   current Fold-0 checkpoint and dataset voxel scale.
#
# Why:
#   Different physical values (e.g. 3.0 vs 4.0 µm) may
#   round to the same integer pooling kernel after model
#   downsampling. We must check this before expensive
#   Fold-0 inference.
#
# Changes files:
#   NO
#
# Runs model inference:
#   NO — only loads the checkpoint/config.
#
# Keep after running:
#   YES — this documents the experimental parameter mapping.
# =========================================================

from pathlib import Path
import sys
import gc

import numpy as np
import pandas as pd
import torch


# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

CKPT = (
    PROJECT
    / "models"
    / "official_baseline"
    / "official_baseline_fold0_best.pth"
)

TRAIN_DIR = (
    PROJECT
    / "data"
    / "raw"
    / "competition"
    / "biohub-cell-tracking-during-development"
    / "train"
)


# ---------------------------------------------------------
# Import paths required by the local MPS adapter
# ---------------------------------------------------------

IMPORT_PATHS = [
    PROJECT / "scripts",
    PROJECT / "external" / "official" / "scripts",
    PROJECT / "external" / "official" / "src",
    PROJECT / "src",
]

for p in IMPORT_PATHS:
    if str(p) not in sys.path:
        sys.path.insert(
            0,
            str(p),
        )


from predict_unet_transformer_mps import (
    load_model,
    pool_kernel_from_um,
)

from tracking_cellmot.io import open_dataset


# ---------------------------------------------------------
# Load checkpoint metadata through the accepted loader
# ---------------------------------------------------------

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

model, window_size, downsample = load_model(
    CKPT,
    device,
)


# ---------------------------------------------------------
# Read actual image voxel scale from one Fold-0 dataset
# ---------------------------------------------------------

sample_name = sorted(
    p.stem
    for p in TRAIN_DIR.glob("*.zarr")
)[0]

sample_ds = open_dataset(
    TRAIN_DIR / sample_name,
    load_image=False,
)

image_scale_um = tuple(
    float(x)
    for x in sample_ds.scale
)

downsample = tuple(
    int(x)
    for x in downsample
)

model_grid_voxel_um = tuple(
    s * d
    for s, d in zip(
        image_scale_um,
        downsample,
    )
)


# ---------------------------------------------------------
# Candidate physical pooling sizes
# ---------------------------------------------------------

candidate_um = [
    1.5,
    2.0,
    2.5,
    3.0,   # current frozen value
    3.5,
    4.0,
    4.5,
    5.0,
]

rows = []

for um in candidate_um:

    kernel = pool_kernel_from_um(
        um,
        model_grid_voxel_um,
    )

    rows.append({
        "pool_kernel_um":
            um,

        "kernel_z":
            int(kernel[0]),

        "kernel_y":
            int(kernel[1]),

        "kernel_x":
            int(kernel[2]),

        "kernel_tuple":
            tuple(
                int(x)
                for x in kernel
            ),

        "same_as_current_3um":
            kernel
            == pool_kernel_from_um(
                3.0,
                model_grid_voxel_um,
            ),
    })


kernel_audit = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("=== MODEL / VOXEL CONFIG ===")

print(
    "Checkpoint:",
    CKPT.name,
)

print(
    "Window size:",
    window_size,
)

print(
    "Model downsample:",
    downsample,
)

print(
    "Image voxel scale (µm):",
    image_scale_um,
)

print(
    "Model-grid voxel scale (µm):",
    model_grid_voxel_um,
)


print(
    "\n=== REALIZED POOLING KERNELS ==="
)

display(
    kernel_audit
)


print(
    "\nUnique kernels:",
    kernel_audit[
        "kernel_tuple"
    ].nunique(),
)


# ---------------------------------------------------------
# Release model memory
# ---------------------------------------------------------

del model

gc.collect()

if (
    torch.backends.mps.is_available()
    and hasattr(torch, "mps")
):
    torch.mps.empty_cache()


print(
    "\n07.09 realized pool-kernel audit: OK"
)

## 07.09 Pool-Kernel Audit — Closed

The model operates on an isotropic 1.625 µm grid.

- 1.5–2.0 µm → `(1,1,1)`: removes local-max suppression and is not a meaningful nearby NMS alternative.
- 2.5–5.0 µm → `(3,3,3)`: identical to the current 3.0 µm setting.

Therefore no Fold-0 pool-kernel inference sweep is required.
The frozen pooling policy remains `(3,3,3)` / `pool_kernel_um=3.0`.

In [ ]:
# =========================================================
# RUN — 07.10 Low-Confidence Node Graph-Role Audit
#
# Purpose:
#   For det=0.995 nodes that do NOT survive det=0.9975,
#   determine where they occur in the tracking graph:
#
#     - singleton
#     - track endpoint
#     - internal continuation
#     - division parent
#     - merge-like / other
#
#   Also measure diagnostic GT-match rate for each role.
#
# Changes files:
#   NO
#
# Uses GT:
#   YES — diagnostic labels only. No GT enters inference.
#
# Keep after running:
#   YES — this is a formal diagnostic experiment.
# =========================================================

role_rows = []

PRED_0995_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0_det09950"
    / "split_0"
)


def classify_graph_role(indegree, outdegree):
    if indegree == 0 and outdegree == 0:
        return "singleton"

    if indegree <= 1 and outdegree <= 1:
        if indegree == 0 or outdegree == 0:
            return "endpoint"
        return "internal"

    if outdegree == 2:
        return "division_parent"

    return "other"


for i, name in enumerate(
    sorted(fold0["dataset"]),
    start=1,
):
    pred_g = load_graph(
        PRED_0995_DIR / f"{name}.geff"
    )

    gt_g = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = np.asarray(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    # -----------------------------------------------------
    # Frozen G2 first
    # -----------------------------------------------------

    apply_frozen_g2(
        pred_g,
        scale,
    )

    # -----------------------------------------------------
    # Diagnostic GT node matching
    # -----------------------------------------------------

    _ = official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(scale),
        max_distance=7.0,
    )

    nodes = (
        pred_g
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        pred_g
        .edge_attrs(unpack=True)
        .to_pandas()
    )

    # -----------------------------------------------------
    # Degrees
    # -----------------------------------------------------

    indegree = (
        edges
        .groupby("target_id")
        .size()
        .to_dict()
    )

    outdegree = (
        edges
        .groupby("source_id")
        .size()
        .to_dict()
    )

    # -----------------------------------------------------
    # Component sizes
    # -----------------------------------------------------

    components = component_sets(
        pred_g
    )

    component_size = {}

    for comp in components:
        n = len(comp)

        for node_id in comp:
            component_size[int(node_id)] = n

    # -----------------------------------------------------
    # High-confidence coordinate set
    # -----------------------------------------------------

    high_conf_set = (
        dataset_key_sets[name][0.9975]
    )

    # -----------------------------------------------------
    # Low-confidence nodes only
    # -----------------------------------------------------

    for node_id, row in nodes.iterrows():

        key = (
            int(row["t"]),
            int(round(row["z"])),
            int(round(row["y"])),
            int(round(row["x"])),
        )

        # Skip nodes surviving the higher threshold
        if key in high_conf_set:
            continue

        node_id = int(node_id)

        indeg = int(
            indegree.get(node_id, 0)
        )

        outdeg = int(
            outdegree.get(node_id, 0)
        )

        comp_size = int(
            component_size[node_id]
        )

        role_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "node_id":
                node_id,

            "component_size":
                comp_size,

            "short_component_le4":
                comp_size <= 4,

            "indegree":
                indeg,

            "outdegree":
                outdeg,

            "graph_role":
                classify_graph_role(
                    indeg,
                    outdeg,
                ),

            "matched_gt_node":
                int(
                    row["match_node_id"]
                ) >= 0,
        })

    if i % 10 == 0:
        print(
            f"Processed {i}/40"
        )


low_conf_role_df = pd.DataFrame(
    role_rows
)

print(
    "\nLow-confidence nodes:",
    len(low_conf_role_df),
)


# =========================================================
# Main role summary
# =========================================================

role_summary = (
    low_conf_role_df
    .groupby(
        [
            "prefix",
            "graph_role",
        ],
        observed=True,
    )
    .agg(
        nodes=(
            "node_id",
            "size",
        ),

        median_component_size=(
            "component_size",
            "median",
        ),

        short_component_rate=(
            "short_component_le4",
            "mean",
        ),

        matched_nodes=(
            "matched_gt_node",
            "sum",
        ),

        matched_rate=(
            "matched_gt_node",
            "mean",
        ),
    )
    .reset_index()
)


print(
    "\n=== LOW-CONFIDENCE NODE ROLE SUMMARY ==="
)

display(
    role_summary
)


# =========================================================
# Long components only: these are exactly the nodes K<=4
# currently preserves.
# =========================================================

long_low_conf = (
    low_conf_role_df[
        low_conf_role_df[
            "component_size"
        ] > 4
    ]
    .copy()
)

long_role_summary = (
    long_low_conf
    .groupby(
        [
            "prefix",
            "graph_role",
        ],
        observed=True,
    )
    .agg(
        nodes=(
            "node_id",
            "size",
        ),

        matched_nodes=(
            "matched_gt_node",
            "sum",
        ),

        matched_rate=(
            "matched_gt_node",
            "mean",
        ),

        median_component_size=(
            "component_size",
            "median",
        ),
    )
    .reset_index()
)


print(
    "\n=== LOW-CONFIDENCE NODES IN COMPONENTS > 4 ==="
)

display(
    long_role_summary
)


print(
    "\n07.10 low-confidence graph-role audit: OK"
)

In [ ]:
# =========================================================
# RUN — 07.11 Low-Confidence Endpoint Trimming Ablation
#
# Purpose:
#   Test whether low-confidence endpoints in components > 4
#   can be removed more selectively than global thresholding.
#
# Frozen baseline-v4:
#   1. det_threshold = 0.995
#   2. G2 division filter
#   3. remove low-confidence nodes if PRE-PRUNING
#      component size <= 4
#   4. singleton cleanup
#   5. remove 2-node components with displacement > 3.5 µm
#
# New experimental addition:
#   AFTER frozen K<=4 confidence pruning,
#   remove low-confidence DEGREE-1 endpoints belonging to
#   components > 4.
#
# Important:
#   - ONE-SHOT only
#   - NO recursive endpoint peeling
#   - division parents are NOT touched
#
# Changes files:
#   NO
#
# Uses GT:
#   YES — official evaluation only.
#   GT does not enter the pruning rule.
#
# Keep after running:
#   YES — formal metric ablation.
# =========================================================

import numpy as np
import pandas as pd


PRED_0995_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0_det09950"
    / "split_0"
)


# =========================================================
# Experimental endpoint-trimming helper
# =========================================================

def apply_low_conf_endpoint_trim(
    graph,
    high_conf_coordinate_set,
):
    """
    One-shot removal of low-confidence degree-1 endpoints
    in components with >4 nodes.

    This is intentionally non-recursive.
    """

    nodes = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        graph
        .edge_attrs(unpack=True)
        .to_pandas()
    )

    indegree = (
        edges
        .groupby("target_id")
        .size()
        .to_dict()
    )

    outdegree = (
        edges
        .groupby("source_id")
        .size()
        .to_dict()
    )

    # -----------------------------------------------------
    # Component size AFTER frozen K<=4 pruning
    # -----------------------------------------------------

    components = component_sets(
        graph
    )

    component_size = {}

    for comp in components:

        n = len(comp)

        for node_id in comp:
            component_size[
                int(node_id)
            ] = n

    remove_nodes = []

    start_endpoints_removed = 0
    end_endpoints_removed = 0

    for node_id, row in nodes.iterrows():

        node_id = int(node_id)

        # Only long components.
        if component_size[node_id] <= 4:
            continue

        # Detection-confidence band.
        key = (
            int(row["t"]),
            int(round(row["z"])),
            int(round(row["y"])),
            int(round(row["x"])),
        )

        is_high_conf = (
            key
            in high_conf_coordinate_set
        )

        if is_high_conf:
            continue

        indeg = int(
            indegree.get(
                node_id,
                0,
            )
        )

        outdeg = int(
            outdegree.get(
                node_id,
                0,
            )
        )

        # -----------------------------------------
        # Degree-1 endpoint only.
        #
        # Start endpoint:
        #   indegree=0, outdegree=1
        #
        # End endpoint:
        #   indegree=1, outdegree=0
        #
        # No singleton and no division parent.
        # -----------------------------------------

        if (
            indeg == 0
            and outdeg == 1
        ):
            remove_nodes.append(
                node_id
            )

            start_endpoints_removed += 1

        elif (
            indeg == 1
            and outdeg == 0
        ):
            remove_nodes.append(
                node_id
            )

            end_endpoints_removed += 1

    if remove_nodes:

        graph.bulk_remove_nodes(
            sorted(
                set(remove_nodes)
            )
        )

    return {
        "nodes_removed":
            len(set(remove_nodes)),

        "start_endpoints_removed":
            start_endpoints_removed,

        "end_endpoints_removed":
            end_endpoints_removed,
    }


# =========================================================
# Evaluate:
#
# baseline_v4
# vs
# baseline_v4 + one-shot endpoint trimming
# =========================================================

VARIANTS = {
    "baseline_v4":
        False,

    "v4_plus_endpoint_trim":
        True,
}

variant_records = {}
dataset_rows_0711 = []


for variant, use_endpoint_trim in VARIANTS.items():

    print(
        "\n" + "=" * 72
    )

    print(
        f"Evaluating: {variant}"
    )

    print(
        "=" * 72
    )

    records = []

    total_conf_k4_removed = 0
    total_endpoint_removed = 0
    total_start_removed = 0
    total_end_removed = 0

    for i, name in enumerate(
        sorted(
            fold0["dataset"]
        ),
        start=1,
    ):

        pred_g = load_graph(
            PRED_0995_DIR
            / f"{name}.geff"
        )

        gt_g = load_graph(
            TRAIN_DIR
            / f"{name}.geff"
        )

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        # -----------------------------------------
        # 1. Frozen G2
        # -----------------------------------------

        apply_frozen_g2(
            pred_g,
            scale,
        )

        high_conf_set = (
            dataset_key_sets[
                name
            ][0.9975]
        )

        # -----------------------------------------
        # 2. Frozen confidence-aware K<=4 pruning
        # -----------------------------------------

        k4_stats = (
            apply_low_conf_component_pruning(
                pred_g,
                high_conf_set,
                4,
            )
        )

        total_conf_k4_removed += (
            k4_stats[
                "low_conf_nodes_removed"
            ]
        )

        # -----------------------------------------
        # 3. Experimental one-shot endpoint trim
        # -----------------------------------------

        endpoint_stats = {
            "nodes_removed": 0,
            "start_endpoints_removed": 0,
            "end_endpoints_removed": 0,
        }

        if use_endpoint_trim:

            endpoint_stats = (
                apply_low_conf_endpoint_trim(
                    pred_g,
                    high_conf_set,
                )
            )

        total_endpoint_removed += (
            endpoint_stats[
                "nodes_removed"
            ]
        )

        total_start_removed += (
            endpoint_stats[
                "start_endpoints_removed"
            ]
        )

        total_end_removed += (
            endpoint_stats[
                "end_endpoints_removed"
            ]
        )

        # -----------------------------------------
        # 4. Frozen graph cleanup
        # -----------------------------------------

        apply_frozen_component_cleanup(
            pred_g,
            scale,
        )

        # -----------------------------------------
        # 5. Locked official metric
        # -----------------------------------------

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(
                estimated_map[name]
            ),
            recall,
        )

        records.append(m)

        dataset_rows_0711.append({
            "variant":
                variant,

            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "endpoint_nodes_removed":
                endpoint_stats[
                    "nodes_removed"
                ],

            "edge_jaccard":
                m["edge_jaccard"],

            "adj_edge_jaccard":
                m["adj_edge_jaccard"],

            "node_recall":
                m["node_recall"],

            "node_ratio":
                m["total_node_ratio"],
        })

        if i % 10 == 0:

            print(
                f"{variant}: "
                f"{i}/40"
            )

    variant_records[
        variant
    ] = records

    print(
        "K<=4 confidence nodes removed:",
        total_conf_k4_removed,
    )

    print(
        "Endpoint nodes removed:",
        total_endpoint_removed,
    )

    print(
        "  start endpoints:",
        total_start_removed,
    )

    print(
        "  end endpoints:",
        total_end_removed,
    )


# =========================================================
# Overall comparison
# =========================================================

overall_rows = []

for variant, records in (
    variant_records.items()
):

    s = summarise(
        records
    )

    overall_rows.append({
        "variant":
            variant,

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m[
                        "total_node_ratio"
                    ]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    })


overall_0711 = pd.DataFrame(
    overall_rows
)


print(
    "\n=== 07.11 OVERALL ENDPOINT-TRIM ABLATION ==="
)

display(
    overall_0711
)


# =========================================================
# Prefix comparison
# =========================================================

names = sorted(
    fold0["dataset"]
)

prefix_rows = []

for variant, records in (
    variant_records.items()
):

    for prefix in [
        "44b6",
        "6bba",
    ]:

        prefix_records = [
            m
            for name, m in zip(
                names,
                records,
            )
            if prefix_map[name]
            == prefix
        ]

        s = summarise(
            prefix_records
        )

        ratios = [
            m["total_node_ratio"]
            for name, m in zip(
                names,
                records,
            )
            if prefix_map[name]
            == prefix
        ]

        prefix_rows.append({
            "variant":
                variant,

            "prefix":
                prefix,

            "edge_jaccard":
                s["edge_jaccard"],

            "adj_edge_jaccard":
                s["adj_edge_jaccard"],

            "division_jaccard":
                s["division_jaccard"],

            "node_recall":
                s["node_recall"],

            "median_node_ratio":
                float(
                    np.median(
                        ratios
                    )
                ),

            "score":
                s["score"],
        })


prefix_0711 = (
    pd.DataFrame(
        prefix_rows
    )
    .sort_values(
        [
            "prefix",
            "variant",
        ]
    )
    .reset_index(drop=True)
)


print(
    "\n=== 07.11 ENDPOINT TRIM BY PREFIX ==="
)

display(
    prefix_0711
)


# =========================================================
# Frozen baseline-v4 reproduction
# =========================================================

baseline_score = float(
    overall_0711.loc[
        overall_0711[
            "variant"
        ] == "baseline_v4",
        "score",
    ].iloc[0]
)

print(
    "\nBaseline-v4 reproduction:",
    baseline_score,
)

print(
    "Difference vs frozen 0.744252:",
    baseline_score
    - 0.744252,
)

print(
    "\n07.11 endpoint-trimming ablation: OK"
)